# RAM on fastMRI brain validation data

The same flow as `RAM_demo.ipynb` -- build a forward operator, generate a
measurement, reconstruct zero-shot, then finetune self-supervised -- but with
**your** data and **your** physics instead of a butterfly and an inpainting
mask:

| demo | here |
|---|---|
| `load_url_image("butterfly.png")` | a real brain slice from the val split |
| `dinv.physics.Inpainting` | `dinv.physics.MultiCoilMRI`, R=8, 20 ACS lines |
| RGB, 3 channels | complex, carried as 2 real channels |
| no reference method | comparable to the LPDSNet / MGLPDSNet cells |

**The point of this notebook is the operator, not the picture.** RAM is only a
fair baseline if `dinv.physics.MultiCoilMRI` is the *same* operator your models
train against. Section 3 checks that against ImMAP's `E` before anything is
reconstructed, and everything downstream is meaningless if it fails.

> Neither `deepinv` nor `ram` was installed when this notebook was written, so
> the API calls below are unverified. They follow the demo's shapes. If a
> signature has moved, section 3 is where you will find out -- it prints what it
> got rather than asserting blindly.

### 1. Install

Same two packages as the demo. `deepinv` supplies the MRI operator; `ram`
supplies the model and the finetuning loop.

In [ ]:
%%capture
pip install git+https://github.com/matthieutrs/ram

### 2. A real brain slice

Read straight from the preprocessed val volume the configs point at -- the same
`image` and `smaps` your models are trained and evaluated on, so the comparison
is against the same ground truth.

In [ ]:
import glob, json, os, sys
import numpy as np
import torch

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.isfile(os.path.join(ROOT, "train.py")):
    ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, ROOT)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

CONFIG = os.path.join(ROOT, "config", "brain", "mg", "mglpds_R8.json")
with open(CONFIG) as f:
    cfg = json.load(f)
val, mri = cfg["data"]["val"], cfg["mri"]
R, ACS = mri["R"], mri["acs_lines"]
print(f"from {os.path.relpath(CONFIG, ROOT)}:  R={R}  acs_lines={ACS}  "
      f"scale_fac={val['scale_fac']}")

smap_root = val["smap_root"]
if not os.path.isabs(smap_root):
    smap_root = os.path.normpath(os.path.join(ROOT, smap_root))
files = sorted(glob.glob(os.path.join(smap_root, "*.h5")))
assert files, f"no volumes under {smap_root}"

SLICE = 4
import h5py
with h5py.File(files[0], "r") as f:
    image = np.asarray(f["image"][SLICE:SLICE + 1])
    smaps = np.asarray(f["smaps"][SLICE:SLICE + 1])

x_true = torch.from_numpy(image).to(torch.complex64).reshape(1, 1, *image.shape[-2:])
smaps = torch.from_numpy(smaps).to(torch.complex64)
smaps = smaps.reshape(1, -1, *smaps.shape[-2:])

# RAM is a foundation model trained on natural images in roughly [0, 1]; the
# preprocessed volumes carry scale_fac (2000 here). Normalise to unit max
# magnitude and keep the factor, so PSNR is computed on the same scale for
# every method rather than on whatever RAM happens to prefer.
SCALE = float(x_true.abs().max())
x_true = (x_true / SCALE).to(device)
smaps = smaps.to(device)

H, W = x_true.shape[-2:]
print(f"slice {SLICE} of {os.path.basename(files[0])}:  {H}x{W}, "
      f"{smaps.shape[1]} coils")
print(f"normalised by max|x| = {SCALE:.4g}")
rss = smaps.abs().pow(2).sum(1).sqrt()
print(f"coil maps unit-RSS on support: max deviation "
      f"{float((rss[rss > 1e-3] - 1).abs().max()):.2e}")

### 3. The forward operator -- and the check everything rests on

Build `dinv.physics.MultiCoilMRI` from the **same** mask and the **same** coil
maps ImMAP uses, then compare it against ImMAP's own
`E = Mask @ FFT2D @ Sense`.

If the two disagree, RAM is being handed a different problem from the one your
models solve and any comparison is void. The usual causes, in order of
likelihood: a different complex layout (deepinv carries complex as 2 real
channels), a different FFT centring convention, or a different normalisation
(`norm="ortho"` versus none).

The cell **reports** rather than asserts, because if it fails you want to see
*how* -- a pure scale factor is a one-line fix, a centring mismatch is not.

In [ ]:
import deepinv as dinv
from operators import FFT2D, Mask, Sense
from physics.mask import make_acc_mask

mask = make_acc_mask((H, W), R, acs_lines=ACS, device=device)
while mask.dim() < 4:
    mask = mask.unsqueeze(0)
print("mask:", tuple(mask.shape), " sampled fraction:", float(mask.mean()))

# ImMAP's operator, the one the trained cells use
E = Mask(mask) @ FFT2D() @ Sense(smaps)

# deepinv's. Complex lives in a 2-channel real axis, so convert on the way in.
def to_dinv(xc):
    """(B, 1, H, W) complex -> (B, 2, H, W) real, deepinv's convention."""
    return torch.cat([xc.real, xc.imag], dim=1)

def from_dinv(xr):
    """(B, 2, H, W) real -> (B, 1, H, W) complex."""
    return torch.complex(xr[:, :1], xr[:, 1:2])

physics = dinv.physics.MultiCoilMRI(
    mask=mask.squeeze(1),            # (B, H, W)
    coil_maps=smaps,
    img_size=(H, W),
    device=device,
)

y_dinv = physics.A(to_dinv(x_true))
y_immap = E(x_true)
print("deepinv y:", tuple(y_dinv.shape), y_dinv.dtype)
print("ImMAP   y:", tuple(y_immap.shape), y_immap.dtype)

# Compare on magnitude, which is layout-agnostic, then on the complex values if
# the layouts line up.
def as_complex_kspace(t):
    if torch.is_complex(t):
        return t
    if t.shape[-3] == 2:                       # (B, C, 2, H, W)
        return torch.complex(t[..., 0, :, :], t[..., 1, :, :])
    if t.shape[1] == 2:                        # (B, 2, H, W)
        return torch.complex(t[:, 0], t[:, 1]).unsqueeze(1)
    raise ValueError(f"unrecognised k-space layout {tuple(t.shape)}")

try:
    a = as_complex_kspace(y_dinv).reshape(-1)
    b = y_immap.reshape(-1)
    if a.numel() == b.numel():
        num = float((a - b).abs().max())
        den = float(b.abs().max()) + 1e-12
        ratio = float((a.abs().sum() / b.abs().sum().clamp_min(1e-12)))
        print(f"\\nmax|A_dinv(x) - E(x)| / max|E(x)| = {num / den:.3e}")
        print(f"energy ratio sum|A_dinv| / sum|E|  = {ratio:.6f}")
        if num / den < 1e-4:
            print("=> SAME OPERATOR. The comparison is valid.")
        elif abs(ratio - 1) > 1e-3:
            print(f"=> differs by a SCALE of about {ratio:.4f}. If that is a"
                  f" sqrt(H*W) or 1/sqrt(H*W) factor it is an ortho-norm"
                  f" mismatch; fix it here, not downstream.")
            print(f"   sqrt(H*W) = {np.sqrt(H * W):.3f}")
        else:
            print("=> same scale but different values: most likely an fftshift"
                  " centring difference. Do NOT proceed until this is resolved.")
    else:
        print(f"\\nelement counts differ ({a.numel()} vs {b.numel()});"
              f" inspect the layouts above.")
except Exception as e:
    print(f"\\ncomparison failed: {type(e).__name__}: {e}")
    print("Inspect the shapes above and adjust as_complex_kspace / to_dinv.")

### 4. The measurement

Noise is added at the grid's own level. `sigma` is the **coil-image** std, which
is what `mri_awgn` uses and what makes it the noise std of the coil-combined
adjoint given unit-RSS maps -- so it means the same thing here as in training.

In [ ]:
from operators.noise import mri_awgn

SIGMA = float(cfg["training"]["val_noise_std"])       # 0.005, the grid's val level
print(f"sigma = {SIGMA}  (config val_noise_std)")

# TWO measurements of the same slice at the same noise level:
#   y_immap  -- ImMAP's simulation, exactly what the trained cells see. Used for
#               the zero-filled reference and the metrics baseline.
#   y_dinv   -- deepinv's, in deepinv's own layout, which is what RAM consumes.
# Section 3 established whether A_dinv and E are the same operator; if it said
# SAME OPERATOR these two differ only by the noise draw.
y_immap, _, _ = mri_awgn(x_true, mask, smaps, SIGMA, "uniform")

physics.noise_model = dinv.physics.GaussianNoise(SIGMA)
y_dinv = physics(to_dinv(x_true))          # calling physics applies A THEN noise

x_zf = E.adjoint(y_immap)                  # zero-filled reference
print("y_immap:", tuple(y_immap.shape), y_immap.dtype)
print("y_dinv :", tuple(y_dinv.shape), y_dinv.dtype)
print("zero-filled:", tuple(x_zf.shape))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, (t, name) in zip(ax, [(x_true, "ground truth"),
                             (x_zf, "zero-filled $E^H y$"),
                             (x_true - x_zf, "difference")]):
    a.imshow(t[0, 0].abs().cpu(), cmap="gray")
    a.set_title(name); a.axis("off")
plt.tight_layout(); plt.show()

### 5. RAM, zero-shot

Exactly the demo's two lines. RAM takes the measurement and the `physics`
object; it is not told anything task-specific beyond that.

In [ ]:
from ram import RAM

model = RAM(device=device)

# RAM consumes the measurement in deepinv's layout, so it gets `y_dinv`. Feeding
# ImMAP's `y_immap` would require guessing that layout; section 3 is what tells
# you the two operators agree, and that is the claim the comparison needs --
# not that the two tensors are byte-identical.
x_hat = model(y_dinv, physics=physics)

x_ram = (from_dinv(x_hat)
         if (not torch.is_complex(x_hat) and x_hat.shape[1] == 2) else x_hat)
print("RAM output:", tuple(x_hat.shape), x_hat.dtype, "-> as complex:",
      tuple(x_ram.shape))

### 6. Score it against the same metrics the grid reports

`training/metrics.py::compute_metrics` is what `eval_mg_recon.py` uses, so these
numbers sit on the same axis as the LPDSNet / MGLPDSNet cells. Magnitude only --
RAM's output and the VarNet baseline are both magnitude, so that is the common
ground.

In [ ]:
from training.metrics import compute_metrics

rows = []
for name, est in (("zero-filled", x_zf), ("RAM zero-shot", x_ram)):
    m = compute_metrics(x_true.abs(), est.abs())
    rows.append((name, {k: float(v) for k, v in m.items()}))

print(f"{'method':<18}{'PSNR':>8}{'SSIM':>8}{'NRMSE':>9}")
for name, m in rows:
    print(f"{name:<18}{m['psnr']:>8.2f}{m['ssim']:>8.4f}{m['nrmse']:>9.4f}")

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
vmax = float(x_true.abs().max())
for a, (t, name) in zip(ax, [(x_true, "ground truth"),
                             (x_zf, "zero-filled"),
                             (x_ram, "RAM zero-shot")]):
    a.imshow(t[0, 0].abs().detach().cpu(), cmap="gray", vmin=0, vmax=vmax)
    a.set_title(name); a.axis("off")
plt.tight_layout(); plt.show()

### 7. Self-supervised finetuning

The demo's last cell, unchanged in spirit: finetune on **this single
measurement**, with no ground truth, using SURE for the Gaussian noise.

This is the part worth taking seriously as a baseline. RAM's claim is not that
it beats a specialist zero-shot -- it is that it gets close *without* task
training, and closes the gap with a handful of self-supervised steps. If a few
steps on one slice moves it materially, that is the interesting result.

`max_iter` is tiny here to keep the cell quick; raise it on a GPU.

In [ ]:
from ram import finetune

model_ft = finetune(RAM(device=device), y_dinv, physics,
                    max_iter=20, noise_loss="SURE", transform="shift",
                    device=device)

with torch.no_grad():
    x_ft_raw = model_ft(y_dinv, physics=physics)
x_ft = from_dinv(x_ft_raw) if (not torch.is_complex(x_ft_raw)
                               and x_ft_raw.shape[1] == 2) else x_ft_raw

m = compute_metrics(x_true.abs(), x_ft.abs())
rows.append(("RAM finetuned", {k: float(v) for k, v in m.items()}))

print(f"{'method':<18}{'PSNR':>8}{'SSIM':>8}{'NRMSE':>9}")
for name, mm in rows:
    print(f"{name:<18}{mm['psnr']:>8.2f}{mm['ssim']:>8.4f}{mm['nrmse']:>9.4f}")

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, (t, name) in zip(ax, [(x_true, "ground truth"), (x_zf, "zero-filled"),
                             (x_ram, "RAM zero-shot"), (x_ft, "RAM finetuned")]):
    a.imshow(t[0, 0].abs().detach().cpu(), cmap="gray", vmin=0, vmax=vmax)
    a.set_title(name); a.axis("off")
plt.tight_layout(); plt.show()

### 8. Before these numbers go next to your own

Four things make this **not yet** a like-for-like comparison:

1. **One slice.** Everything above is a single volume, single slice. The grid
   evaluates over the whole val split at a fixed seed. Loop this before
   quoting a number.

2. **RAM is given the coil maps; so are your unrolled nets; E2E-VarNet is
   not.** Three different amounts of prior knowledge across the three arms.
   Fine, as long as it is stated.

3. **Magnitude only.** RAM's output goes through `.abs()` here, as VarNet's
   does. Your LPDS cells reconstruct complex, so they are being scored on less
   than they produce.

4. **The normalisation in section 2 is load-bearing.** RAM is trained on images
   in roughly [0, 1]; the volumes carry `scale_fac = 2000`. If you change how
   `SCALE` is computed, RAM's numbers move and your models' do not -- so keep it
   identical across every arm, or normalise none of them.

A fifth, if section 3 reported anything other than "SAME OPERATOR": fix that
first. Every number above is conditional on it.